[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/07_Building_Knowledge_Graphs.ipynb)

> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 构建知识图谱（Building Knowledge Graphs）

## 概览

本 notebook 演示如何用 Semantica 的图谱构建模块，把抽取出的实体和关系组装成知识图谱。你将学会使用 `GraphBuilder` 和 `EntityResolver`。

**文档**：[API 参考](https://semantica.readthedocs.io/reference/kg/)

### 先看主线

前两篇备好了原料：实体（圈好的名字）、关系（画好的线）、三元组（定型的档案卡）。本篇开始组装成图，四步分两段：

**把图搭起来（第 1–2 步）—— 组装**

1. 第 1 步 备料：抽取实体提及（mention）和关系
2. 第 2 步 组装：给每个提及分配图 ID，按关系的真实端点建边

**把图理干净（第 3–4 步）—— 去重**

3. 第 3 步 实体消解：`EntityResolver` 一步到位，合并重复提及、重映射边的端点
4. 第 4 步 精细去重：`deduplication` 模块拆成检测与合并两步，控制更细

留意一个伏笔：第 1 步的示例文本故意让 `Apple Inc.` 出现两次，第 2 步建出来的图因此会有两个 Apple 节点——第 3、4 步的两种方案，解决的都是这个问题。

### 学习目标

- 抽取实体提及与关系，并把它们映射成图谱记录
- 使用 `GraphBuilder` 构建一张边来自实际抽取关系的图
- 使用 `EntityResolver` 合并重复提及并重映射关系端点
- 使用 `semantica.deduplication` 模块，并给出完整的去重后实体集

## 安装

从 PyPI 安装 Semantica：

```bash
pip install semantica
# 或安装全部可选依赖：
pip install semantica[all]
```

---

## 第 1 步：抽取实体和关系

从文本中抽取实体提及和关系。示例文本在两个句子里分别提到 `Apple Inc.`，这样后面就能演示重复提及如何被消解成同一个规范实体。

In [ ]:
%pip install semantica

# spaCy models are distributed separately from the spaCy library. This lesson
# relies on the English model to recognize standalone places such as Cupertino.
import sys
import subprocess
import spacy

try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


In [ ]:
from semantica.semantic_extract import NERExtractor, RelationExtractor

text = (
    "Apple Inc. is headquartered in Cupertino, California. "
    "Tim Cook is the CEO of Apple Inc. "
    "The company is a technology company."
)

ner_extractor = NERExtractor()
relation_extractor = RelationExtractor()

mentions = ner_extractor.extract(text)
relations = relation_extractor.extract(text, mentions)

print("Entity mentions:")
for mention in mentions:
    print(f"  {mention.text!r:<13} {mention.label:<7} span=[{mention.start_char}:{mention.end_char}]")

print("\nExtracted relations:")
for rel in relations:
    print(f"  {rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")

## 第 2 步：构建知识图谱

备好的料要拼成图。给每个提及分配一个图 ID，再把每条关系的 `subject` 和 `object` 换算成这些 ID。关键在最后一句：边必须从关系的真实端点映射而来，而不是按列表位置猜端点——这是"图忠实于文本"的保证。

In [ ]:
from semantica.kg import GraphBuilder

entities = []
span_to_id = {}
for i, mention in enumerate(mentions, 1):
    graph_id = f"e{i}"
    span_to_id[(mention.start_char, mention.end_char)] = graph_id
    entities.append({
        "id": graph_id,
        "type": mention.label,
        "name": mention.text,
        "properties": {},
    })

relationships = []
for rel in relations:
    source_id = span_to_id.get((rel.subject.start_char, rel.subject.end_char))
    target_id = span_to_id.get((rel.object.start_char, rel.object.end_char))
    if source_id is None or target_id is None:
        print(f"Skipping relation with unmapped endpoint: "
              f"{rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")
        continue
    relationships.append({
        "source": source_id,
        "target": target_id,
        "type": rel.predicate,
        "properties": {},
    })

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Graph entities ({len(knowledge_graph['entities'])}):")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']})")

print(f"\nGraph relationships ({len(knowledge_graph['relationships'])}):")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

edges = {
    (id_to_name[r["source"]], r["type"], id_to_name[r["target"]])
    for r in knowledge_graph["relationships"]
}
assert ("Apple Inc.", "located_in", "Cupertino") in edges
assert ("Tim Cook", "works_for", "Apple Inc.") in edges

## 第 3 步：实体消解

伏笔在这里兑现：图里同一家组织现在有两个节点。`EntityResolver` 把重复提及合并成一个规范实体（canonical entity），并记录哪些源 ID 被合并（`merged_from`），边的端点随之重映射到规范实体上。合并有据可查，不是静默覆盖。

In [ ]:
from semantica.kg import EntityResolver

entity_resolver = EntityResolver()
resolved_entities = entity_resolver.resolve_entities(entities)

canonical_id = {}
for entity in resolved_entities:
    for source_id in entity.get("merged_from", [entity["id"]]):
        canonical_id[source_id] = entity["id"]
    if entity.get("merged_from"):
        print(f"Merged {entity['merged_from']} -> {entity['id']}: {entity['name']}")

print(f"\nMentions in: {len(entities)}, resolved entities out: {len(resolved_entities)}")

resolved_names = {entity["id"]: entity["name"] for entity in resolved_entities}
print("\nRelationships remapped onto canonical entities:")
for relationship in relationships:
    source = canonical_id[relationship["source"]]
    target = canonical_id[relationship["target"]]
    print(f"  {resolved_names[source]} --{relationship['type']}--> {resolved_names[target]}")

canonical_entities = {(entity["name"], entity["type"]) for entity in resolved_entities}
assert canonical_entities == {
    ("Apple Inc.", "ORG"),
    ("Tim Cook", "PERSON"),
    ("Cupertino", "GPE"),
    ("California", "GPE"),
}
assert len(resolved_entities) == 4

## 第 4 步：去重

同一个问题，换个工具从细处做。`semantica.deduplication` 模块对合并过程提供更细的控制。注意一个易错点：`merge_duplicates` 按"重复组"返回 `MergeOperation`，每组一个——完整的去重集合，是这些合并结果**加上**所有没进任何组的实体，两者缺一不可。

In [ ]:
from semantica.deduplication import DuplicateDetector, EntityMerger, MergeStrategy

detector = DuplicateDetector(similarity_threshold=0.8)
duplicate_groups = detector.detect_duplicate_groups(entities)
print(f"Duplicate groups: {len(duplicate_groups)}")
for group in duplicate_groups:
    print(f"  {[entity['name'] for entity in group.entities]} "
          f"(confidence={group.confidence:.2f})")

merger = EntityMerger()
merge_operations = merger.merge_duplicates(
    entities, strategy=MergeStrategy.KEEP_MOST_COMPLETE
)

merged_source_ids = {
    entity["id"] for op in merge_operations for entity in op.source_entities
}
untouched_entities = [e for e in entities if e["id"] not in merged_source_ids]
deduplicated_entities = untouched_entities + [
    op.merged_entity for op in merge_operations
]

print(f"\nMerge operations: {len(merge_operations)}")
print(f"Deduplicated entities ({len(deduplicated_entities)}):")
for entity in deduplicated_entities:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']})")

assert len(merge_operations) == 1
assert len(deduplicated_entities) == 4

## 小结

### 回顾主线

本篇四步两段：**组装**（备料、按真实端点建边）搭出一张忠实于文本的图，**去重**（实体消解、精细去重）把同一实体的多个节点归一。前一段解决"图建得对"，后一段解决"图理得净"。

### 你学到了什么

你已学会如何构建知识图谱：

- **从抽取到图谱**：把每个提及映射成图 ID，边从 `Relation.subject` / `Relation.object` 的真实端点构建
- **GraphBuilder**：从显式的 `{"entities": ..., "relationships": ...}` 输入构建知识图谱
- **EntityResolver**：把重复提及合并成规范实体，并重映射关系端点
- **去重**：把 `MergeOperation` 的结果与未受影响的实体合并，得到完整的去重集合

下一步：在 Graph_Analytics notebook 中学习如何分析图谱。